# Sesión 21 — Ética, Equidad, Privacidad y Despliegue Clínico
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo VI · Integración, Ética y Cierre**

## Objetivos de aprendizaje

1. Cuantificar el **sesgo algorítmico** en ML de salud y rastrear su origen en los datos de entrenamiento.
2. Comprender la **deriva de dataset** (covariable, etiqueta, concepto) y medir su impacto en la práctica.
3. Implementar una simulación de **aprendizaje federado** y comprender sus garantías de privacidad.
4. Aplicar **privacidad diferencial** vía ruido de gradiente y cuantificar el compromiso privacidad-utilidad.
5. Navegar el panorama regulatorio: clasificación FDA SaMD, IEC 62304, y monitoreo de modelos.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Obermeyer, Z. et al. (2019). Dissecting racial bias in an algorithm used to manage the health of populations. *Science*, 366(6464). |
| ★★★ | Subbaswamy, A. & Saria, S. (2020). From development to deployment: dataset shift, causality, and shift-stable models in health AI. *Biostatistics*, 21(2). |
| ★★☆ | McMahan, B. et al. (2017). Communication-efficient learning of deep networks from decentralized data (FedAvg). *AISTATS*. |
| ★★☆ | Abadi, M. et al. (2016). Deep learning with differential privacy. *ACM CCS*. |
| ★★☆ | FDA (2021). AI/ML-based Software as a Medical Device (SaMD) Action Plan. FDA.gov. |
| ★☆☆ | Wiens, J. et al. (2019). Do no harm: a roadmap for responsible machine learning for health care. *Nature Medicine*, 25. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

rng = np.random.default_rng(42)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

## Parte 1 — Sesgo algorítmico: cuando los proxies engañan

El artículo de Obermeyer et al. (2019) encontró que un algoritmo clínico ampliamente
desplegado usaba el **costo de atención médica** como proxy de la **necesidad** de
atención médica — subestimando sistemáticamente la severidad de la enfermedad en
pacientes afroamericanos, quienes recibían menos atención al mismo nivel de costo
debido a barreras históricas de acceso.

Simulamos un escenario análogo: un modelo de riesgo de readmisión entrenado en una
característica proxy sesgada.

In [ ]:
# Simular un dataset de pacientes de 2 grupos
# Desenlace verdadero: readmisión a 30 días (necesidad clínica)
# Proxy (sesgado): número de visitas previas a especialista (depende del acceso)
n_A, n_B = 600, 400   # grupo A (mayoría), grupo B (acceso históricamente menor)

def crear_pacientes(n, tasa_readmision, escala_visitas, offset_comorbilidad, rng_):
    readmite     = rng_.binomial(1, tasa_readmision, n)
    edad         = rng_.normal(65 + offset_comorbilidad*2, 12, n)
    comorbilidad = rng_.poisson(2 + offset_comorbilidad, n)
    # Proxy: visitas correlacionadas con readmisión PERO también con acceso (escala_visitas)
    visitas      = rng_.poisson((1 + readmite) * escala_visitas, n)
    return np.column_stack([edad, comorbilidad, visitas]), readmite

Xa, ya = crear_pacientes(n_A, tasa_readmision=0.20, escala_visitas=3.0, offset_comorbilidad=0,   rng_=rng)
Xb, yb = crear_pacientes(n_B, tasa_readmision=0.28, escala_visitas=1.5, offset_comorbilidad=0.8, rng_=rng)
# Grupo B: mayor tasa de readmisión real pero menos visitas (barrera de acceso)

X_all = np.vstack([Xa, Xb]).astype(np.float32)
y_all = np.hstack([ya, yb]).astype(np.int64)
grupo = np.hstack([np.zeros(n_A), np.ones(n_B)]).astype(int)

nombres_feat = ['Edad', 'Comorbilidades', 'Visitas previas (proxy)']

scaler = StandardScaler().fit(X_all)
X_s = scaler.transform(X_all)

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X_s, y_all, grupo, test_size=0.3, stratify=y_all, random_state=0)

# Entrenar dos modelos: uno con proxy, uno sin
clf_sesgado    = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
clf_no_sesgado = LogisticRegression(max_iter=500).fit(X_tr[:, :2], y_tr)  # sin la característica de visitas

def metricas_equidad(clf, X_te, y_te, g_te, usar_visitas=True):
    Xe = X_te if usar_visitas else X_te[:, :2]
    p  = clf.predict_proba(Xe)[:, 1]
    # Umbral en la prevalencia
    thr  = y_te.mean()
    pred = (p >= thr).astype(int)
    resultados = {}
    for g, gname in [(0, 'Grupo A'), (1, 'Grupo B')]:
        mask = g_te == g
        cm   = confusion_matrix(y_te[mask], pred[mask])
        tn, fp, fn, tp = cm.ravel()
        resultados[gname] = {
            'AUROC':         roc_auc_score(y_te[mask], p[mask]),
            'Sensibilidad':  tp / (tp+fn),
            'TFP':           fp / (fp+tn),
            'Prevalencia':   y_te[mask].mean(),
        }
    return resultados

res_sesgado    = metricas_equidad(clf_sesgado,    X_te, y_te, g_te, usar_visitas=True)
res_no_sesgado = metricas_equidad(clf_no_sesgado, X_te, y_te, g_te, usar_visitas=False)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metricas  = ['AUROC', 'Sensibilidad', 'TFP']
grupos    = ['Grupo A', 'Grupo B']

for ax, metric in zip(axes, metricas):
    x_pos = np.arange(len(grupos))
    b1 = [res_sesgado[g][metric]    for g in grupos]
    b2 = [res_no_sesgado[g][metric] for g in grupos]
    w  = 0.3
    ax.bar(x_pos - w/2, b1, w, label='Con proxy (visitas)', color='tomato',    alpha=0.8)
    ax.bar(x_pos + w/2, b2, w, label='Sin proxy',           color='steelblue', alpha=0.8)
    ax.set(xticks=x_pos, xticklabels=grupos, title=metric, ylabel=metric)
    ax.legend(fontsize=8)

plt.suptitle('Sesgo algorítmico: la característica proxy (visitas previas) codifica desigualdad de acceso\n'
             'El Grupo B tiene mayor tasa de readmisión real pero menor AUROC con el modelo sesgado',
             y=1.01)
plt.tight_layout()
plt.show()

print('Tasas de readmisión reales:')
print(f'  Grupo A: {ya.mean():.2%}   Grupo B: {yb.mean():.2%}  (Grupo B tiene mayor necesidad)')
print('\nCon proxy (visitas):    ', {g: f'AUROC={res_sesgado[g]["AUROC"]:.3f}' for g in grupos})
print('Sin proxy:              ', {g: f'AUROC={res_no_sesgado[g]["AUROC"]:.3f}' for g in grupos})

## Parte 2 — Deriva de dataset: medición y mitigación

In [ ]:
# Simular deriva de covariables: entrenar en Hospital A, desplegar en Hospital B
# El Hospital B tiene pacientes más viejos y más enfermos (distribución de entrada distinta)

def crear_datos_hospital(n, edad_media, comorbilidad_media, tasa_readmision, rng_):
    edad   = rng_.normal(edad_media, 12, n).astype(np.float32)
    comor  = rng_.poisson(comorbilidad_media, n).astype(np.float32)
    fc     = rng_.normal(75 + 0.1*edad, 12, n).astype(np.float32)
    creat  = rng_.gamma(1.5 + 0.02*comor, 0.5, n).astype(np.float32)
    y      = rng_.binomial(1, np.clip(tasa_readmision + 0.005*(edad-65) + 0.03*comor, 0.05, 0.95), n)
    return np.column_stack([edad, comor, fc, creat]), y

# Fuente: Hospital A (sitio de entrenamiento)
X_A, y_A = crear_datos_hospital(800, edad_media=58, comorbilidad_media=1.8,
                                  tasa_readmision=0.15, rng_=rng)
# Objetivo: Hospital B (sitio de despliegue — más viejos, más enfermos)
X_B, y_B = crear_datos_hospital(400, edad_media=68, comorbilidad_media=3.1,
                                  tasa_readmision=0.25, rng_=rng)
# Objetivo: Hospital C (similar a A)
X_C, y_C = crear_datos_hospital(300, edad_media=59, comorbilidad_media=2.0,
                                  tasa_readmision=0.16, rng_=rng)

sc_shift = StandardScaler().fit(X_A)
X_A_s = sc_shift.transform(X_A).astype(np.float32)
X_B_s = sc_shift.transform(X_B).astype(np.float32)
X_C_s = sc_shift.transform(X_C).astype(np.float32)

# Entrenar en A, evaluar en A, B, C
clf_shift = LogisticRegression(max_iter=500)
clf_shift.fit(X_A_s[:600], y_A[:600])

sitios = [('Hospital A (fuente)',   X_A_s[600:], y_A[600:], 'steelblue'),
          ('Hospital B (con deriva)', X_B_s,       y_B,       'tomato'),
          ('Hospital C (similar)',  X_C_s,       y_C,       'seagreen')]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

aurocs, etiquetas, colores_bar = [], [], []
for name, X_site, y_site, color in sitios:
    p    = clf_shift.predict_proba(X_site)[:, 1]
    auroc = roc_auc_score(y_site, p)
    aurocs.append(auroc); etiquetas.append(name); colores_bar.append(color)
    print(f'{name:32s}  AUROC={auroc:.3f}  prevalencia={y_site.mean():.2%}')

axes[0].bar(etiquetas, aurocs, color=colores_bar, alpha=0.8, edgecolor='white')
axes[0].set(ylabel='AUROC', title='Deriva de dataset: modelo entrenado en Hospital A\n'
                                    'El rendimiento se degrada en el sitio B con deriva',
            ylim=(0.5, 1.0))
axes[0].tick_params(axis='x', labelrotation=20)
for i, v in enumerate(aurocs):
    axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=9)

# Visualización PCA de la deriva de distribución
from sklearn.decomposition import PCA
pca_shift = PCA(n_components=2).fit(X_A_s)
for X_site, y_site, name, color in [
    (X_A_s, y_A, 'Hospital A', 'steelblue'),
    (X_B_s, y_B, 'Hospital B', 'tomato'),
]:
    X2d = pca_shift.transform(X_site)
    axes[1].scatter(X2d[:,0], X2d[:,1], alpha=0.3, s=10,
                     color=color, label=name)

axes[1].set(xlabel='PC1', ylabel='PC2',
            title='Deriva de covariables visualizada (PCA)\n'
                  'Los pacientes del Hospital B ocupan un espacio de características distinto')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Parte 3 — Simulación de aprendizaje federado (FedAvg)

In [ ]:
# FedAvg: cada cliente entrena localmente, el servidor promedia los pesos —
# los datos nunca salen del sitio

class SimpleNet(nn.Module):
    def __init__(self, in_dim=4, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, 16),     nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x): return self.net(x).squeeze(1)

def entrenar_local(model, X_np, y_np, n_epochs=5, lr=1e-3):
    model.train()
    Xt = torch.tensor(X_np, dtype=torch.float32)
    yt = torch.tensor(y_np, dtype=torch.float32)
    opt = optim.Adam(model.parameters(), lr=lr)
    for _ in range(n_epochs):
        opt.zero_grad()
        F.binary_cross_entropy_with_logits(model(Xt), yt).backward()
        opt.step()
    return {k: v.clone() for k, v in model.state_dict().items()}

def fedavg(pesos_clientes, tamanos_clientes):
    total = sum(tamanos_clientes)
    avg   = {}
    for key in pesos_clientes[0]:
        avg[key] = sum(w[key] * (n/total) for w, n in zip(pesos_clientes, tamanos_clientes))
    return avg

# Tres hospitales, los datos permanecen locales
clientes = [
    (X_A_s[:600],  y_A[:600].astype(np.float32)),
    (X_B_s,        y_B.astype(np.float32)),
    (X_C_s,        y_C.astype(np.float32)),
]
X_test_fed = np.vstack([X_A_s[600:], X_B_s[:100]])
y_test_fed = np.hstack([y_A[600:],   y_B[:100]])

# Línea base centralizada (suponiendo que la concentración de datos fuera posible)
X_central = np.vstack([c[0] for c in clientes])
y_central = np.hstack([c[1] for c in clientes])
net_central = SimpleNet()
entrenar_local(net_central, X_central, y_central, n_epochs=50)
with torch.no_grad():
    p_central = torch.sigmoid(net_central(torch.tensor(X_test_fed, dtype=torch.float32))).numpy()
auroc_central = roc_auc_score(y_test_fed, p_central)

# Entrenamiento federado
modelo_global = SimpleNet()
auroc_fed_rondas = []
n_rondas = 30

for rnd in range(n_rondas):
    pesos_clientes, tamanos_clientes = [], []
    for X_c, y_c in clientes:
        modelo_local = SimpleNet()
        modelo_local.load_state_dict(modelo_global.state_dict())
        w = entrenar_local(modelo_local, X_c, y_c, n_epochs=3)
        pesos_clientes.append(w)
        tamanos_clientes.append(len(X_c))

    modelo_global.load_state_dict(fedavg(pesos_clientes, tamanos_clientes))

    modelo_global.eval()
    with torch.no_grad():
        p_fed = torch.sigmoid(modelo_global(torch.tensor(X_test_fed, dtype=torch.float32))).numpy()
    auroc_fed_rondas.append(roc_auc_score(y_test_fed, p_fed))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(auroc_fed_rondas, 'b-', lw=2, label='Federado (FedAvg)')
ax.axhline(auroc_central, color='tomato', ls='--', lw=2,
            label=f'Cota superior centralizada ({auroc_central:.3f})')
ax.set(xlabel='Ronda de comunicación', ylabel='AUROC',
       title='Aprendizaje federado (FedAvg) — 3 hospitales\nLos datos nunca salen de cada sitio')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f'AUROC federado final:     {auroc_fed_rondas[-1]:.3f}')
print(f'Cota superior centralizada: {auroc_central:.3f}')
print(f'Costo de privacidad (brecha AUROC): {auroc_central - auroc_fed_rondas[-1]:.3f}')

## Parte 4 — Privacidad diferencial: ruido de gradiente

In [ ]:
# Privacidad diferencial vía recorte de gradiente + ruido gaussiano (DP-SGD)
# Presupuesto de privacidad: ε controla el compromiso privacidad-utilidad

def entrenar_con_dp(X_np, y_np, noise_multiplier=1.0, max_grad_norm=1.0,
                     n_epochs=40, lr=1e-3, batch_size=64):
    model = SimpleNet()
    opt   = optim.Adam(model.parameters(), lr=lr)
    Xt    = torch.tensor(X_np, dtype=torch.float32)
    yt    = torch.tensor(y_np, dtype=torch.float32)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    for ep in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            # Gradientes por muestra (DP-SGD lo requiere)
            clipped_grads = [torch.zeros_like(p) for p in model.parameters()]

            for xi, yi in zip(Xb, yb):
                opt.zero_grad()
                loss_i = F.binary_cross_entropy_with_logits(
                    model(xi.unsqueeze(0)), yi.unsqueeze(0))
                loss_i.backward()
                # Recortar el gradiente por muestra
                total_norm = torch.sqrt(sum(p.grad.norm()**2
                                            for p in model.parameters() if p.grad is not None))
                clip_factor = min(1.0, max_grad_norm / (total_norm + 1e-8))
                for p, cg in zip(model.parameters(), clipped_grads):
                    if p.grad is not None:
                        cg += p.grad * clip_factor

            # Añadir ruido gaussiano y aplicar
            opt.zero_grad()
            for p, cg in zip(model.parameters(), clipped_grads):
                noise = torch.randn_like(cg) * noise_multiplier * max_grad_norm
                p.grad = (cg + noise) / len(Xb)
            opt.step()

    return model


niveles_ruido = [0.0, 0.5, 1.0, 2.0, 4.0]   # 0 = sin DP
auroc_dp = []

X_dp  = X_A_s[:400]
y_dp  = y_A[:400].astype(np.float32)

for sigma in niveles_ruido:
    m = entrenar_con_dp(X_dp, y_dp, noise_multiplier=sigma, n_epochs=30)
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(torch.tensor(X_A_s[600:], dtype=torch.float32))).numpy()
    auroc_dp.append(roc_auc_score(y_A[600:], p))
    print(f'σ={sigma:.1f}  AUROC={auroc_dp[-1]:.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(niveles_ruido, auroc_dp, 'b-o', lw=2.5, ms=8)
ax.set(xlabel='Multiplicador de ruido σ  (mayor = más privado, menos exacto)',
       ylabel='AUROC', title='Compromiso privacidad-utilidad (DP-SGD)\n'
              'σ=0: sin privacidad.  σ>1: garantía de DP significativa.')
ax.annotate('Sin privacidad', (0, auroc_dp[0]), fontsize=8, xytext=(0.05, auroc_dp[0]+0.01))
ax.annotate('Privacidad fuerte', (niveles_ruido[-1], auroc_dp[-1]),
             fontsize=8, xytext=(niveles_ruido[-1]-1.0, auroc_dp[-1]+0.015))
plt.tight_layout()
plt.show()

## Parte 5 — Lista de verificación regulatoria y plantilla de tarjeta de modelo

In [ ]:
# Lista de verificación de preparación regulatoria para un sistema de ML clínico
lista_verificacion = {
    'Clínico y científico': [
        ('Población de uso previsto definida',          True),
        ('Desenlace clínicamente significativo elegido', True),
        ('Dataset de validación externa obtenido',       False),
        ('Rendimiento por subgrupo reportado',           True),
        ('Calibración evaluada',                         True),
        ('Análisis de curva de decisión realizado',      False),
    ],
    'Técnico': [
        ('Pipeline de entrenamiento reproducible',       True),
        ('Versionado de modelo implementado',            True),
        ('Auditoría de fuga de datos completada',        True),
        ('Robustez adversarial probada',                 False),
        ('Método de explicabilidad validado',            True),
        ('Cuantificación de incertidumbre implementada', False),
    ],
    'Regulatorio y ético': [
        ('Clase de riesgo SaMD FDA determinada',         True),
        ('Ciclo de vida de software IEC 62304 documentado', False),
        ('Auditoría de sesgo entre grupos demográficos', True),
        ('Gobernanza de datos / aprobación de comité de ética', True),
        ('Desidentificación de PHI verificada',          True),
        ('Plan de monitoreo post-mercado definido',      False),
    ],
}

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for ax, (categoria, items) in zip(axes, lista_verificacion.items()):
    etiquetas_c = [item[0] for item in items]
    hecho       = [item[1] for item in items]
    colores_c   = ['seagreen' if d else 'tomato' for d in hecho]
    ax.barh(etiquetas_c, [1]*len(items), color=colores_c, alpha=0.8, edgecolor='white')
    for i, (label, d) in enumerate(zip(etiquetas_c, hecho)):
        ax.text(0.5, i, '✅' if d else '❌', ha='center', va='center',
                 fontsize=12, fontweight='bold')
    n_hecho = sum(hecho)
    ax.set(xlim=(0, 1), xticks=[], title=f'{categoria}\n{n_hecho}/{len(items)} completo',
           yticks=range(len(items)), yticklabels=etiquetas_c)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Lista de Verificación de Preparación Regulatoria para ML Clínico\n'
             'Verde = completo, Rojo = aún se requiere antes del despliegue', y=1.01)
plt.tight_layout()
plt.show()

total_hecho = sum(v for items in lista_verificacion.values() for _, v in items)
total_items = sum(len(items) for items in lista_verificacion.values())
print(f'Preparación general: {total_hecho}/{total_items} ({100*total_hecho/total_items:.0f}%)')
print('\nElementos pendientes:')
for cat, items in lista_verificacion.items():
    for label, hecho in items:
        if not hecho:
            print(f'  [{cat}] {label}')

## ✏️ Ejercicios

1. **Taxonomía de métricas de equidad.** Implementa cuatro criterios de equidad —
   paridad demográfica, igualdad de odds, igualdad de oportunidad, y paridad predictiva —
   sobre el modelo sesgado de readmisión. Demuestra que es matemáticamente imposible
   satisfacer los cuatro simultáneamente cuando las tasas base difieren (Chouldechova
   2017). ¿Qué criterio es más apropiado para una herramienta de riesgo de readmisión,
   y por qué?

2. **Corrección de deriva de covariables por ponderación de importancia.** Estima el
   ratio de densidad $w(x) = p_B(x)/p_A(x)$ usando una regresión logística entrenada
   para discriminar entre muestras del Hospital A y B. Usa estos pesos durante el
   entrenamiento en el Hospital A para reponderar la pérdida. Mide la mejora en el
   AUROC del Hospital B.

3. **FedAvg con datos no-IID.** Repite el experimento de aprendizaje federado pero haz
   que la distribución de clases sea muy distinta entre clientes (Hospital A: 10% de
   readmisión, Hospital B: 40%, Hospital C: 20%). Compara la convergencia de FedAvg con
   FedProx (añade un término proximal $\frac{\mu}{2}\|w - w^{global}\|^2$ a la pérdida
   de cada cliente). ¿Por qué FedAvg tiene dificultades con datos no-IID?

4. **Contabilidad del presupuesto de privacidad.** Implementa un contador de Privacidad
   Diferencial de Rényi (RDP) para calcular el presupuesto de privacidad (ε, δ)
   consumido tras $T$ pasos de entrenamiento con tamaño de batch $q$ y multiplicador de
   ruido σ. Grafica ε vs número de épocas de entrenamiento para σ ∈ {1.0, 2.0, 4.0} e
   interpreta los resultados clínicamente.

5. *(Desafío)* **Monitoreo de deriva de modelo.** Tras el despliegue, un modelo clínico
   debe monitorearse por degradación de rendimiento. Implementa un monitor de AUROC con
   ventana deslizante: simula 500 predicciones secuenciales de pacientes, inyecta una
   deriva de rendimiento gradual en el paciente 300 (debido a un cambio de protocolo), e
   implementa una carta de control CUSUM para detectar la deriva automáticamente.
   Calcula el retraso promedio de detección y la tasa de falsas alarmas.

## 📚 Recursos

| Recurso | Fuente | Notas |
|---|---|
| Aequitas (toolkit de equidad) | https://github.com/dssg/aequitas | Auditoría de sesgo en ML |
| Opacus (DP-SGD) | https://opacus.ai | Librería de entrenamiento DP para PyTorch |
| PySyft (federado) | https://github.com/OpenMined/PySyft | Framework de aprendizaje federado |
| Guía FDA SaMD | https://www.fda.gov/medical-devices/software-medical-device-samd | |
| Obermeyer et al. 2019 | https://doi.org/10.1126/science.aax2342 | Sesgo racial en algoritmo clínico |